<a href="https://colab.research.google.com/github/AzharJodatti/pytorch-experiments/blob/main/pytorch_device_reproducibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Device management, timing & reproducibility

The "plumbing" topics every PyTorch user eventually gets bitten by.

In [pytorch_basics.ipynb](https://github.com/AzharJodatti/pytorch-experiments/blob/main/pytorch_basics.ipynb) the exercises already touched `torch.manual_seed`, `torch.cuda.manual_seed` and moving tensors to the GPU with `.to(device)`. This notebook zooms in on those details:

1. **Device management** — tensors vs models, and the classic *"Expected all tensors to be on the same device"* error
2. **Timing training** — `time.time()` vs `torch.cuda.Event`
3. **Reproducibility** — a `set_seeds()` helper (torch + cuda + numpy)
4. **CPU vs GPU determinism** — why same-seed reruns can still differ
5. **dtype deep dive** — `float32` vs `float16` vs `int64`, memory per element, mismatch errors

In [ ]:
import torch
import numpy as np
from torch import nn
import matplotlib.pyplot as plt

print(f"torch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

torch version: 2.14.0+cpu
Using device: cpu


## 1. Device management: `.to(device)` for tensors AND models

`device = "cuda" if torch.cuda.is_available() else "cpu"` is the one-liner you should put at the top of every notebook. Then:

- **Tensors:** `t = t.to(device)` (returns a *new* tensor on that device — tensors are immutable, so you must reassign)
- **Models:** `model = model.to(device)` (moves all parameters/buffers in-place, but you still reassign by convention)

PyTorch will **not** move anything for you. If one tensor lives on the CPU and another on the GPU, the operation fails. That's the single most common beginner error.

In [ ]:
cpu_tensor = torch.tensor([1.0, 2.0, 3.0])
device_tensor = cpu_tensor.to(device)   # same call works for tensors...
print(f"cpu_tensor on:      {cpu_tensor.device}")
print(f"device_tensor on:   {device_tensor.device}")

cpu_tensor on:      cpu
device_tensor on:   cpu


### The classic error: forgetting `.to(device)`

Below we **deliberately** try to multiply a CPU tensor with a CUDA tensor. If you're on a GPU machine this raises the famous error. On a CPU-only machine (like this one) there is no second device to mismatch, so the cell prints what the error looks like instead — either way you see the message.

In [ ]:
if device == "cuda":
    try:
        cpu_t = torch.randn(2, 3)                    # lives on cpu
        gpu_t = torch.randn(3, 2, device="cuda")     # lives on cuda
        _ = cpu_t @ gpu_t                            # boom
    except RuntimeError as e:
        print("Caught the classic device-mismatch error:")
        print(e)
else:
    # No GPU here, so we can't genuinely mix devices.
    # On a GPU machine `cpu_t @ gpu_t` raises exactly this:
    print("CPU-only machine: no cuda device to mismatch with.")
    print("On a GPU box, cpu_t @ gpu_t raises:")
    print('  RuntimeError: Expected all tensors to be on the same device '
          '(but found at least two devices, cpu and cuda:0)!')

CPU-only machine: no cuda device to mismatch with.
On a GPU box, cpu_t @ gpu_t raises:
  RuntimeError: Expected all tensors to be on the same device (but found at least two devices, cpu and cuda:0)!


The same trap applies to **models**: if you forget `model.to(device)` but did `.to(device)` on the data, the forward pass crashes with the very same error — the model's weights are just tensors living on the CPU. Rule of thumb: create `device` once, and send *both* the model and every batch to it.

## 2. Timing training: `time.time()` vs `torch.cuda.Event`

Two ways to time a training loop:

- **`time.time()`** — wall-clock around the whole loop. Simple, works everywhere.
- **`torch.cuda.Event` + `torch.cuda.synchronize()`** — the GPU way. CUDA kernels run *asynchronously*: by the time Python reaches the end of the loop, the GPU may still be working. `time.time()` can therefore report a too-optimistic number. CUDA events are recorded into the GPU's own stream, and `synchronize()` blocks until the GPU actually finishes, giving the true GPU time.

We run the **same** small regression experiment (`y = 0.7x + 0.3`, a tiny MLP, 500 epochs — same data as `pytorch_basics.ipynb`) twice, once with each method. On a CPU-only machine the CUDA-event run gracefully falls back to `time.time()`.

In [ ]:
# ---- shared experiment setup: y = 0.7x + 0.3, same data as pytorch_basics ----
WEIGHT, BIAS = 0.7, 0.3

X = torch.arange(0.0, 1.0, 0.02).unsqueeze(dim=1)        # shape (50, 1)
y = WEIGHT * X + BIAS

train_split = int(0.8 * len(X))
X_train, y_train = X[:train_split].to(device), y[:train_split].to(device)
X_test,  y_test  = X[train_split:].to(device),  y[train_split:].to(device)


class TinyMLP(nn.Module):
    """Small MLP in the style of the models in this repo."""
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=1, out_features=16)
        self.layer_2 = nn.Linear(in_features=16, out_features=16)
        self.layer_3 = nn.Linear(in_features=16, out_features=1)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))


def train_model(epochs: int = 500):
    """Fresh model + 500 epochs of the usual train loop. Returns (model, final_loss)."""
    model = TinyMLP().to(device)
    loss_fn = nn.L1Loss()
    optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)

    for epoch in range(epochs):
        model.train()
        y_pred = model(X_train)
        loss = loss_fn(y_pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model, loss.item()


# a quick sanity check of the experiment itself
torch.manual_seed(42)
model_sanity, sanity_loss = train_model(epochs=500)
print(f"sanity run -> final L1 loss: {sanity_loss:.6f}  (target: learn y = 0.7x + 0.3)")
print(f"sanity run -> first test prediction: {model_sanity(X_test[:1]).item():.4f} (should be ~{WEIGHT*X_test[0].item()+BIAS:.4f})")

sanity run -> final L1 loss: 0.013532  (target: learn y = 0.7x + 0.3)
sanity run -> first test prediction: 0.8305 (should be ~0.8600)


In [ ]:
# ---- Run A: wall-clock timing with time.time() ----
import time

torch.manual_seed(42)
t0 = time.time()
model_a, loss_a = train_model(epochs=500)
wall_seconds = time.time() - t0

print(f"Run A (time.time() wall-clock): {wall_seconds:.4f} seconds for 500 epochs, final loss {loss_a:.6f}")

Run A (time.time() wall-clock): 0.2317 seconds for 500 epochs, final loss 0.013532


In [ ]:
# ---- Run B: torch.cuda.Event + synchronize(), with a graceful CPU fallback ----
def train_and_time_with_events(epochs: int = 500):
    """Times training with CUDA events when a GPU is present; falls back to time.time()."""
    if torch.cuda.is_available():
        try:
            torch.manual_seed(42)
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)

            model = TinyMLP().to(device)
            loss_fn = nn.L1Loss()
            optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)

            start_event.record()
            for epoch in range(epochs):
                model.train()
                y_pred = model(X_train)
                loss = loss_fn(y_pred, y_train)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            end_event.record()

            # wait until the GPU has actually finished everything above
            torch.cuda.synchronize()
            elapsed_ms = start_event.elapsed_time(end_event)
            return model, loss.item(), elapsed_ms / 1000.0, "torch.cuda.Event (GPU timing)"
        except Exception as e:
            print(f"CUDA event timing failed ({e!r}) -> falling back to time.time()")

    # CPU fallback: CUDA events don't exist without a GPU
    torch.manual_seed(42)
    t0 = time.time()
    model, loss = train_model(epochs=epochs)
    return model, loss, time.time() - t0, "time.time() wall-clock fallback (no GPU present)"


model_b, loss_b, seconds_b, method_b = train_and_time_with_events(epochs=500)
print(f"Run B timing method used: {method_b}")
print(f"Run B: {seconds_b:.4f} seconds for 500 epochs, final loss {loss_b:.6f}")

# the two runs used the same seed and the same data -> identical results
print(f"Run A vs Run B final loss equal: {loss_a == loss_b}")

Run B timing method used: time.time() wall-clock fallback (no GPU present)
Run B: 0.2575 seconds for 500 epochs, final loss 0.013532
Run A vs Run B final loss equal: True


On a GPU box, Run B's event-based number is the honest one — `time.time()` around an async loop can under-count. On CPU both methods measure the same thing, which is exactly what the fallback prints.

## 3. Reproducibility deep dive: `set_seeds()`

PyTorch initializes weights randomly. **No seed → every run starts from different random numbers → different weights → different losses.** That makes debugging (and comparing experiments) miserable.

A good `set_seeds()` helper covers *all* the random number generators your pipeline uses:

- `torch.manual_seed(seed)` — CPU (and, conveniently, all CUDA devices too in recent PyTorch)
- `torch.cuda.manual_seed_all(seed)` — explicit belt-and-braces for every GPU (matters in multi-GPU setups)
- `np.random.seed(seed)` — **numpy has its own RNG!** `sklearn`/numpy data prep (train/test splits, shuffling, noise) does not care about your torch seed, so it must be seeded separately

In [ ]:
def set_seeds(seed: int):
    """Seed every RNG a typical PyTorch pipeline uses: torch (cpu), all gpus, numpy."""
    torch.manual_seed(seed)          # cpu rng (also seeds all cuda devices in modern torch)
    torch.cuda.manual_seed_all(seed) # explicit: every cuda device, belt-and-braces
    np.random.seed(seed)             # numpy/sklearn data prep has its own rng

print("set_seeds() defined")

set_seeds() defined


In [ ]:
# Demo 1: without a seed, two freshly created models start from DIFFERENT weights
m1, m2 = TinyMLP(), TinyMLP()
w1 = next(m1.parameters()).detach()[0, 0].item()
w2 = next(m2.parameters()).detach()[0, 0].item()
print(f"unseeded model 1 first weight: {w1:+.6f}")
print(f"unseeded model 2 first weight: {w2:+.6f}")
print(f"same? {w1 == w2}   <- pure luck if they ever match")

unseeded model 1 first weight: -0.206077
unseeded model 2 first weight: +0.873114
same? False   <- pure luck if they ever match


In [ ]:
# Demo 2: with set_seeds(), two fresh models start IDENTICALLY
set_seeds(42)
m1 = TinyMLP()
w1 = next(m1.parameters()).detach()[0, 0].item()

set_seeds(42)
m2 = TinyMLP()
w2 = next(m2.parameters()).detach()[0, 0].item()

print(f"seeded model 1 first weight: {w1:+.6f}")
print(f"seeded model 2 first weight: {w2:+.6f}")
print(f"same? {w1 == w2}   <- reproducible!")

seeded model 1 first weight: +0.764539
seeded model 2 first weight: +0.764539
same? True   <- reproducible!


In [ ]:
# Demo 3: numpy's RNG is a separate universe — seed it too
np.random.seed(42)
a = np.random.rand(3)
b = np.random.rand(3)

np.random.seed(42)
c = np.random.rand(3)

print(f"numpy without re-seeding, two draws: {a} vs {b} -> different: {not np.allclose(a, b)}")
print(f"numpy re-seeded, two draws:         {a} vs {c} -> identical: {np.allclose(a, c)}")

numpy without re-seeding, two draws: [0.37454012 0.95071431 0.73199394] vs [0.59865848 0.15601864 0.15599452] -> different: True
numpy re-seeded, two draws:         [0.37454012 0.95071431 0.73199394] vs [0.37454012 0.95071431 0.73199394] -> identical: True


Practical rule: call `set_seeds(42)` right **before** model creation *and* before any data prep (splits, shuffling, augmentation) you want reproducible.

## 4. CPU vs GPU determinism: seeds are not the whole story

Even with the same seed, **same-seed reruns can still differ slightly on GPU**. Why:

- Many GPU kernels (e.g. the atomicAdd-based ones, and parallel **reductions** like `sum`) are **non-deterministic**: floating-point addition is not associative, so the *order* in which thousands of threads add numbers changes the last bits of the result.
- `cudnn.benchmark` picks different (fastest) convolution algorithms run-to-run.
- TF32 / reduced-precision matmul (see `torch.get_float32_matmul_precision()` below) trades precision for speed.

PyTorch offers `torch.use_deterministic_algorithms(True)` to force deterministic kernels (plus the `CUBLAS_WORKSPACE_CONFIG` env var on CUDA ≥ 10.2) — it guarantees reproducibility but can be slower or raise errors for ops with no deterministic implementation. We won't enable it here; the light-touch version is `torch.backends.cudnn.deterministic = True`, which makes cuDNN pick deterministic algorithms.

In [ ]:
# light-touch determinism settings (harmless no-ops on CPU-only machines)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False    # stop cudnn from auto-picking different algorithms

print(f"torch.backends.cudnn.deterministic : {torch.backends.cudnn.deterministic}")
print(f"torch.backends.cudnn.benchmark     : {torch.backends.cudnn.benchmark}")
print(f"torch.get_float32_matmul_precision(): {torch.get_float32_matmul_precision()}")

torch.backends.cudnn.deterministic : True
torch.backends.cudnn.benchmark     : False
torch.get_float32_matmul_precision(): highest


In [ ]:
# On CPU, same seed -> bit-for-bit reproducible training. Check it:
set_seeds(42)
_, loss_r1 = train_model(epochs=500)
set_seeds(42)
_, loss_r2 = train_model(epochs=500)

print(f"CPU rerun 1 final loss: {loss_r1:.10f}")
print(f"CPU rerun 2 final loss: {loss_r2:.10f}")
print(f"bit-for-bit identical: {loss_r1 == loss_r2}")
print("(on GPU these can differ in the last decimals for the reasons above)")

CPU rerun 1 final loss: 0.0135320276
CPU rerun 2 final loss: 0.0135320276
bit-for-bit identical: True
(on GPU these can differ in the last decimals for the reasons above)


## 5. dtype deep dive: `float32` vs `float16` vs `int64`

Every tensor has a `dtype`, and it decides both **precision** and **memory**:

| dtype | name | bytes/element | typical use |
|---|---|---|---|
| `torch.float32` | single precision | 4 | **PyTorch's default** for weights/inputs |
| `torch.float16` | half precision | 2 | GPU speedups / mixed precision |
| `torch.int64` | long | 8 | labels & indices (default for integer tensors) |

`.element_size()` gives the bytes per element — multiply by `.numel()` for total bytes.

In [ ]:
t_float32 = torch.randn(3, 3)                      # default dtype
t_float16 = torch.randn(3, 3, dtype=torch.float16) # explicit dtype at creation
t_int64   = torch.arange(6).reshape(3, 2)          # integer tensors default to int64

for name, t in [("float32 (default)", t_float32), ("float16", t_float16), ("int64", t_int64)]:
    print(f"{name:20s} dtype={str(t.dtype):15s} element_size={t.element_size()} bytes "
          f"-> total {t.numel() * t.element_size()} bytes for {t.numel()} elements")

# float16 halves memory vs float32: same shape, half the bytes
print(f"\nfloat32 total vs float16 total: {t_float32.numel()*t_float32.element_size()} vs {t_float16.numel()*t_float16.element_size()} bytes")

float32 (default)    dtype=torch.float32   element_size=4 bytes -> total 36 bytes for 9 elements
float16              dtype=torch.float16   element_size=2 bytes -> total 18 bytes for 9 elements
int64                dtype=torch.int64     element_size=8 bytes -> total 48 bytes for 6 elements

float32 total vs float16 total: 36 vs 18 bytes


In [ ]:
# dtype conversions with torch.tensor(..., dtype=...) and .to(dtype)
t = torch.tensor([1, 2, 3], dtype=torch.float16)   # create with a dtype
t_f64 = t.to(torch.float64)                        # convert after creation
t_back = t_f64.to(torch.int64)                     # ...and back (values truncate)

print(f"t       : dtype={t.dtype}, values={t.tolist()}")
print(f"t_f64   : dtype={t_f64.dtype}, values={t_f64.tolist()}")
print(f"t_back  : dtype={t_back.dtype}, values={t_back.tolist()}  <- 1.5 would become 1")
print(f"model parameters are float32: {next(TinyMLP().parameters()).dtype}")

t       : dtype=torch.float16, values=[1.0, 2.0, 3.0]
t_f64   : dtype=torch.float64, values=[1.0, 2.0, 3.0]
t_back  : dtype=torch.int64, values=[1, 2, 3]  <- 1.5 would become 1
model parameters are float32: torch.float32


### The classic dtype error: feeding `float64` where the model wants `float32`

`torch.tensor([...])` built from **Python floats** becomes `float64` (numpy's default leaks through), while a model's parameters are `float32`. Feed a `float64` tensor into a `float32` model and PyTorch refuses rather than silently converting:

In [ ]:
loss_fn = nn.L1Loss()
model = TinyMLP().to(device)

pred   = torch.tensor([[0.1], [0.2], [0.3]])                 # float32 -> model is happy
target = torch.tensor([[0.1], [0.2], [0.3]],
                      dtype=torch.float64)                   # oops: python-float tensor is float64

try:
    out = model(target)                                      # boom
    print("no error?!", out)
except RuntimeError as e:
    print("Caught the classic dtype-mismatch error:")
    print(e)

# the fix: cast the input to the model's dtype
out = model(target.to(torch.float32))
print(f"\nafter target.to(torch.float32), forward pass works: {out.detach().flatten().tolist()}")
print(f"loss against float32 targets: {loss_fn(out, torch.zeros_like(out)).item():.4f} (dtype={loss_fn(out, torch.zeros_like(out)).dtype})")

Caught the classic dtype-mismatch error:
mat1 and mat2 must have the same dtype, but got Double and Float

after target.to(torch.float32), forward pass works: [0.18838858604431152, 0.19141441583633423, 0.20034462213516235]
loss against float32 targets: 0.1934 (dtype=torch.float32)


## What we learned

- **Device management:** `device = "cuda" if torch.cuda.is_available() else "cpu"`, then `.to(device)` for *both* tensors (`t = t.to(device)`) and models (`model = model.to(device)`). Forgetting one side raises `Expected all tensors to be on the same device`.
- **Timing:** `time.time()` is fine on CPU, but GPU kernels run asynchronously — `torch.cuda.Event` + `torch.cuda.synchronize()` measures the GPU's true time. Always write the GPU path inside a `try` with a `time.time()` fallback so the code runs anywhere (this notebook used the fallback: **CPU-only sandbox**).
- **Reproducibility:** one helper, `set_seeds(seed)`, seeding `torch.manual_seed`, `torch.cuda.manual_seed_all` **and** `np.random.seed` (numpy/sklearn data prep has its own RNG). Without it two models get different initial weights; with it they match bit-for-bit.
- **Determinism:** same seed ≠ guaranteed identical GPU results — non-deterministic kernels and parallel reduction order change the last float bits. `torch.backends.cudnn.deterministic = True` helps; `torch.use_deterministic_algorithms(True)` is the heavy hammer. This sandbox reports `float32_matmul_precision = highest` (CPU default).
- **dtypes:** default `float32` (4 bytes/element), `float16` (2 bytes — half the memory), `int64` (8 bytes, used for labels). A `float64` input into a `float32` model raises `mat1 and mat2 must have the same dtype` — fix with `.to(torch.float32)`.